In [ ]:
import os
"""
Build Raw_Disease_List.csv — UMLS CUI grouping prerequisite
=============================================================
Strategy (no UMLS API key required):
  Step 1. Extract CUI→name pairs from already-parsed source files.
          Coverage: ~85% of all CUIs from internal files alone.
  Step 2. For remaining ~15% (NOT_FOUND), query Gemini in batches.
          Gemini knows UMLS concept names from its training data.
 
Source file roles:
  ChEMBL    : umls_cui + mesh_heading (single per row, most reliable)
  DrugBank  : UMLS_CUI | Standard_Disease_Names (positional pairing)
  IUPHAR    : UMLS_CUI | Standard_Disease_Names (positional pairing)
  TTD       : *_UMLS | *_Diseases columns (positional, strip ICD suffix)
  DrugCentral: OMOP/UMLS count mismatches → CUIs in master list, names skipped
 
Output: Raw_Disease_List.csv
  Columns: umls_cui | disease_name | source_hint
"""
 
import pandas as pd
import re
import time
import json
import google.generativeai as genai
from pathlib import Path
 
# ── Config: adjust paths to your environment ─────────────────────────────────
GOOGLE_API_KEY = os.environ.get("GEMINI_API_KEY")
 
BASE = Path("./Output/DB")
SOURCE_FILES = {
    "ChEMBL":      BASE / "ChEMBL/NAR/ChEMBL_v36_Indications_Standardized.csv",
    "DrugBank":    BASE / "DrugBank/NAR/DrugBank_Master_Standardized.csv",
    "IUPHAR":      BASE / "IUPHAR/NAR/IUPHAR_GPCR_Indications_Standardized.csv",
    "TTD":         BASE / "TTD/NAR/TTD_Master_Standardized.csv",
}
DRUG_IND_PATH    = BASE / "GPCRactDB/Drug_Indication_Master_Integrated.csv"
TARGET_IND_PATH  = BASE / "GPCRactDB/Target_Indication_Master_Integrated.csv"
OUTPUT_PATH      = BASE / "GPCRactDB/Raw_Disease_List.csv"
CHECKPOINT_PATH  = BASE / "GPCRactDB/raw_disease_checkpoint.csv"
 
GEMINI_BATCH_SIZE = 30      # CUIs per Gemini call
CHECKPOINT_EVERY  = 10      # batches between checkpoint saves
 
 

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def valid_cui(s: str) -> bool:
    """Standard UMLS CUI format: C followed by 7 digits."""
    return bool(re.match(r"^C\d{7}$", str(s).strip()))


def clean_ttd_name(raw: str) -> str:
    """Strip ICD-11 suffix from TTD disease names.
    Input:  'Bipolar disorder [ICD-11: 6A60]'
    Output: 'Bipolar disorder'
    """
    return re.sub(r"\s*[\[\(]ICD-11:.*?[\]\)]", "", str(raw)).strip()


def extract_pairs_positional(df: pd.DataFrame,
                              name_col: str, cui_col: str,
                              source_tag: str,
                              clean_fn=None) -> tuple[list, int]:
    """
    Split both columns on '|' and pair positionally.
    Only processes rows where name count == CUI count.
    Returns (list of (cui, name, source) tuples, n_skipped).
    """
    pairs, skipped = [], 0
    for _, row in df.iterrows():
        names_raw = str(row.get(name_col) or "")
        cuis_raw  = str(row.get(cui_col)  or "")
        if names_raw in ("", "nan") or cuis_raw in ("", "nan"):
            continue
        names = [n.strip() for n in names_raw.split("|")]
        cuis  = [c.strip() for c in cuis_raw.split("|")]
        if len(names) != len(cuis):
            skipped += 1
            continue
        for name, cui in zip(names, cuis):
            if not valid_cui(cui):
                continue
            if clean_fn:
                name = clean_fn(name)
            if name and name != "nan":
                pairs.append((cui, name, source_tag))
    return pairs, skipped


def build_lookup_from_files(source_files: dict) -> dict:
    """
    Load all source files and build {umls_cui: (disease_name, source)} lookup.
    Priority order: ChEMBL > DrugBank > IUPHAR > TTD (first match wins).
    """
    lookup = {}

    # --- ChEMBL: single CUI + mesh_heading per row ---------------------------
    path = source_files.get("ChEMBL")
    if path and Path(path).exists():
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            cui  = str(row.get("umls_cui", "") or "").strip()
            name = str(row.get("mesh_heading", "") or "").strip()
            if valid_cui(cui) and name and name != "nan":
                lookup.setdefault(cui, (name, "ChEMBL"))
        print(f"  ChEMBL   : {sum(1 for v in lookup.values() if v[1]=='ChEMBL'):,} pairs")

    # --- DrugBank: UMLS_CUI | Standard_Disease_Names (positional) -----------
    path = source_files.get("DrugBank")
    if path and Path(path).exists():
        df = pd.read_csv(path)
        pairs, skip = extract_pairs_positional(
            df, "Standard_Disease_Names", "UMLS_CUI", "DrugBank"
        )
        added = sum(1 for cui, name, src in pairs
                    if cui not in lookup and not lookup.setdefault(cui, (name, src)) or False)
        # simpler approach:
        added = 0
        for cui, name, src in pairs:
            if cui not in lookup:
                lookup[cui] = (name, src)
                added += 1
        print(f"  DrugBank : +{added:,} (skipped {skip} mismatched rows)")

    # --- IUPHAR: UMLS_CUI | Standard_Disease_Names (positional) -------------
    path = source_files.get("IUPHAR")
    if path and Path(path).exists():
        df = pd.read_csv(path)
        pairs, _ = extract_pairs_positional(
            df, "Standard_Disease_Names", "UMLS_CUI", "IUPHAR"
        )
        added = 0
        for cui, name, src in pairs:
            if cui not in lookup:
                lookup[cui] = (name, src)
                added += 1
        print(f"  IUPHAR   : +{added:,}")

    # --- TTD: two column pairs, strip ICD suffix from names ------------------
    path = source_files.get("TTD")
    if path and Path(path).exists():
        df = pd.read_csv(path)
        pairs_d, _ = extract_pairs_positional(
            df, "Drug_Clinical_Indications", "Drug_Clinical_Indications_UMLS",
            "TTD_Drug", clean_fn=clean_ttd_name
        )
        pairs_t, _ = extract_pairs_positional(
            df, "Target_Biological_Diseases", "Target_Biological_Diseases_UMLS",
            "TTD_Target", clean_fn=clean_ttd_name
        )
        added = 0
        for cui, name, src in pairs_d + pairs_t:
            if cui not in lookup:
                lookup[cui] = (name, src)
                added += 1
        print(f"  TTD      : +{added:,}")

    print(f"  Total from files: {len(lookup):,} unique CUI→name pairs")
    return lookup


def query_gemini_for_names(model, cuis: list[str]) -> dict:
    """
    Ask Gemini to return the standard English disease name for each UMLS CUI.
    Returns {cui: disease_name} for CUIs Gemini recognizes.
    """
    prompt = (
        "You are a medical ontology expert. "
        "Return the standard English disease/condition name for each UMLS CUI below.\n"
        "Respond ONLY with a JSON object: {CUI: name}. "
        "Use \"UNKNOWN\" for CUIs you are not confident about. "
        "No markdown, no explanation.\n\n"
        "CUIs:\n" + "\n".join(cuis)
    )
    try:
        response = model.generate_content(prompt)
        raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", response.text.strip())
        result = json.loads(raw)
        return {k: v for k, v in result.items() if v != "UNKNOWN"}
    except Exception as e:
        print(f"    [WARN] Gemini batch failed: {e}")
        return {}


def chunk_list(lst, size):
    return [lst[i:i+size] for i in range(0, len(lst), size)]



In [ ]:
# ── Main ─────────────────────────────────────────────────────────────────────
def main():
    print("=== Step 1: Load master CUI list ===")
    drug_df   = pd.read_csv(DRUG_IND_PATH)
    target_df = pd.read_csv(TARGET_IND_PATH)
    all_cuis  = sorted(set(
        drug_df["UMLS_CUI"].dropna().tolist() +
        target_df["UMLS_CUI"].dropna().tolist()
    ))
    valid_cuis   = [c for c in all_cuis if valid_cui(c)]
    invalid_cuis = [c for c in all_cuis if not valid_cui(c)]
    print(f"  Valid CUIs  : {len(valid_cuis):,}")
    print(f"  Invalid/skip: {len(invalid_cuis)} → {invalid_cuis[:5]}")

    print("\n=== Step 2: Build name lookup from source files ===")
    lookup = build_lookup_from_files(SOURCE_FILES)

    not_found = [c for c in valid_cuis if c not in lookup]
    print(f"\n  Coverage: {len(valid_cuis)-len(not_found):,}/{len(valid_cuis):,} "
          f"({(1-len(not_found)/len(valid_cuis))*100:.1f}%) from files")
    print(f"  Remaining for Gemini: {len(not_found):,}")

    print("\n=== Step 3: Fill gaps with Gemini ===")

    # Load checkpoint
    if Path(CHECKPOINT_PATH).exists():
        ckpt = pd.read_csv(CHECKPOINT_PATH)
        done = {row["umls_cui"]: row["disease_name"]
                for _, row in ckpt.iterrows() if row["disease_name"] != "NOT_FOUND"}
        lookup.update({k: (v, "Gemini") for k, v in done.items()})
        not_found = [c for c in not_found if c not in lookup]
        print(f"  Checkpoint loaded: {len(done):,} already resolved | remaining: {len(not_found):,}")

    if not_found and GOOGLE_API_KEY != "YOUR_GOOGLE_API_KEY_HERE":
        genai.configure(api_key=GOOGLE_API_KEY)
        model = genai.GenerativeModel(
            model_name="gemini-2.5-flash-lite",
            generation_config=genai.GenerationConfig(temperature=0.0),
        )
        batches = chunk_list(not_found, GEMINI_BATCH_SIZE)
        gemini_results = {}

        for i, batch in enumerate(batches):
            result = query_gemini_for_names(model, batch)
            gemini_results.update(result)
            for cui, name in result.items():
                lookup[cui] = (name, "Gemini")
            time.sleep(0.1)

            if (i + 1) % CHECKPOINT_EVERY == 0:
                # Save intermediate checkpoint
                ckpt_records = [{"umls_cui": c, "disease_name": v, "source_hint": "Gemini"}
                                for c, v in gemini_results.items()]
                pd.DataFrame(ckpt_records).to_csv(CHECKPOINT_PATH, index=False)
                print(f"  Batch {i+1}/{len(batches)} | checkpoint saved | found so far: {len(gemini_results):,}")

        print(f"  Gemini resolved: {len(gemini_results):,} additional CUIs")
    elif not_found:
        print("  [SKIP] Gemini key not set — NOT_FOUND entries will remain as-is")

    print("\n=== Step 4: Build Raw_Disease_List.csv ===")
    records = []
    for cui in valid_cuis:
        if cui in lookup:
            name, src = lookup[cui]
        else:
            name, src = "NOT_FOUND", "NONE"
        records.append({"umls_cui": cui, "disease_name": name, "source_hint": src})

    # Add invalid CUIs for completeness
    for cui in invalid_cuis:
        records.append({"umls_cui": cui, "disease_name": "INVALID_FORMAT", "source_hint": "NONE"})

    final_df = pd.DataFrame(records)
    not_found_n = (final_df["disease_name"] == "NOT_FOUND").sum()
    found_n = len(final_df) - not_found_n - len(invalid_cuis)

    print(f"  Total rows  : {len(final_df):,}")
    print(f"  Found       : {found_n:,} ({found_n/len(final_df)*100:.1f}%)")
    print(f"  NOT_FOUND   : {not_found_n:,} ({not_found_n/len(final_df)*100:.1f}%)")
    print(f"\n  Source breakdown:")
    print(final_df["source_hint"].value_counts().to_string())

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\n  Saved: {OUTPUT_PATH}")

    # Quick sample
    print("\n  Sample (found):")
    print(final_df[final_df["disease_name"] != "NOT_FOUND"].head(8)
          [["umls_cui","disease_name","source_hint"]].to_string(index=False))



In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
import os
"""
Build UMLS_CUI_to_Category.csv
================================
Assigns each UMLS CUI to one of 12 GPCR-relevant therapeutic categories.

Method (hybrid, in priority order):
  1. Rule-based keyword mapping (instant, no API):
     Fast pre-filter for clearly identifiable diseases.
  2. Gemini LLM classification (remaining):
     Batched API calls using disease_name text.
  3. Fallback: "Other / Unclassified" for Unknown/NOT_FOUND entries.

The 12 therapeutic categories are defined for GPCR drug research,
aligned with major GPCR-targeted drug classes. They are not strictly
WHO-official but follow EMA therapeutic area conventions commonly used
in pharmacology literature. Cite in methods as:
  "GPCR-relevant therapeutic area classification (our definition)"

Input : Raw_Disease_List.csv  (umls_cui | disease_name | source_hint)
Output: UMLS_CUI_to_Category.csv  (umls_cui | disease_name | category_id | category_name)
"""

import pandas as pd
import google.generativeai as genai
import json
import re
import time
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
GOOGLE_API_KEY   = os.environ.get("GEMINI_API_KEY")
INPUT_PATH       = "./Output/DB/GPCRactDB/Raw_Disease_List.csv"
OUTPUT_PATH      = "./Output/DB/GPCRactDB/UMLS_CUI_to_Category.csv"
CHECKPOINT_PATH  = "./Output/DB/GPCRactDB/category_checkpoint.csv"
GEMINI_BATCH     = 25
CHECKPOINT_EVERY = 10   # batches

# ── 12 therapeutic categories ─────────────────────────────────────────────────
CATEGORIES = {
    1:  "CNS / Psychiatric",
    2:  "Cardiovascular",
    3:  "Metabolic / Endocrine",
    4:  "Inflammatory / Immune",
    5:  "Respiratory",
    6:  "Oncology / Cancer",
    7:  "Renal / Urological",
    8:  "Gastrointestinal",
    9:  "Pain / Musculoskeletal",
    10: "Reproductive / Hormonal",
    11: "Infectious Disease",
    12: "Other / Unclassified",
}

# ── Rule-based keyword pre-filter ────────────────────────────────────────────
# Maps lowercase keyword → category_id. Applied before Gemini.
# Covers ~60-70% of common diseases instantly.
KEYWORD_RULES = [
    # CNS / Psychiatric (1)
    (1, ["schizophrenia","psychosis","depression","bipolar","alzheimer","parkinson",
         "epilep","seizure","dementia","anxiety","autism","adhd","attention deficit",
         "multiple sclerosis","huntington","migraine","neuropath","neurodegenerat",
         "stroke","cerebral","brain","spinal","narcolep","insomnia","sleep disorder"]),
    # Cardiovascular (2)
    (2, ["hypertension","heart failure","arrhythmia","coronary","angina","atherosclerosis",
         "myocardial infarct","cardiac","atrial fibrill","ventricular","aortic",
         "cardiovascular","thrombosis","thromboembol","anticoagulant","platelet",
         "peripheral arterial","ischemia","cardio"]),
    # Metabolic / Endocrine (3)
    (3, ["diabetes","obesity","insulin","thyroid","adrenal","hypoglycemia","hyperlipidemia",
         "dyslipidemia","metabolic syndrome","cushing","acromegaly","hyperglycemia",
         "glucagon","glucos","lipid","cholesterol","hypercholesterol","fatty liver",
         "non-alcoholic","pituitary","growth hormone"]),
    # Inflammatory / Immune (4)
    (4, ["rheumatoid arthritis","lupus","crohn","inflammatory bowel","multiple sclerosis",
         "psoriasis","atopic","allerg","autoimmune","immune","immunodeficiency",
         "sarcoidosis","vasculitis","gout","systemic lupus","ankylosing","uveitis",
         "eczema","dermatitis"]),
    # Respiratory (5)
    (5, ["asthma","copd","pulmonary","bronch","cystic fibrosis","pneumonia","lung disease",
         "emphysema","respiratory","idiopathic pulmonary fibrosis","sarcoidosis",
         "rhinitis","sinusitis","nasal"]),
    # Oncology / Cancer (6)
    (6, ["cancer","carcinoma","tumor","tumour","neoplasm","lymphoma","leukemia","melanoma",
         "sarcoma","glioma","glioblastoma","myeloma","oncol","metastatic","malignant",
         "adenocarcinoma","hepatocellular"]),
    # Renal / Urological (7)
    (7, ["kidney","renal","nephro","dialysis","bladder","urinary","incontinence",
         "overactive bladder","benign prostatic","prostatic","ureter","urethra",
         "glomerulo","polycystic kidney"]),
    # Gastrointestinal (8)
    (8, ["irritable bowel","gastroparesis","constipation","diarrhea","nausea","vomiting",
         "ulcerative colitis","colitis","gastric","peptic ulcer","gastroesophageal",
         "gastrointestinal","bowel","hepat","liver fibrosis","biliary","pancreatitis",
         "cholestasis"]),
    # Pain / Musculoskeletal (9)
    (9, ["pain","analges","opioid","fibromyalgia","osteoarthritis","osteoporosis",
         "bone","musculoskeletal","arthritis","gout","fracture","tendon","muscle",
         "spasm","nociception","neuropathic pain","chronic pain","low back"]),
    # Reproductive / Hormonal (10)
    (10, ["endometriosis","infertility","preterm","labor","menopause","polycystic ovary",
          "contracepti","amenorrhea","uterine","ovarian","prostate hypertrophy",
          "testosterone","estrogen","menstrual","dysmenorrhea","obstetric",
          "pregnancy","preeclampsia","oxytocin","gnrh","lh receptor","fsh receptor"]),
    # Infectious Disease (11)
    (11, ["hiv","aids","hepatitis","covid","coronavirus","malaria","tuberculosis",
          "bacterial","fungal","viral","infection","sepsis","pneumococcal",
          "influenza","herpes","cmv","ebola","syphilis","chlamydia"]),
]

def rule_based_category(name: str) -> int | None:
    """
    Apply keyword rules to disease name.
    Returns category_id (1-12) if matched, None otherwise.
    Checks CNS last to avoid false positives from 'neurological' in other conditions.
    """
    name_lower = name.lower()
    for cat_id, keywords in KEYWORD_RULES:
        if any(kw in name_lower for kw in keywords):
            return cat_id
    return None


def chunk_list(lst, size):
    return [lst[i:i+size] for i in range(0, len(lst), size)]



In [ ]:
GEMINI_SYSTEM = """You are a pharmacology expert specializing in GPCR drug classification.
Assign each disease name to exactly ONE of these 12 therapeutic categories:

1: CNS / Psychiatric  (Schizophrenia, Depression, Alzheimer's, Parkinson's, Epilepsy, Migraine...)
2: Cardiovascular  (Hypertension, Heart failure, Arrhythmia, Coronary artery disease...)
3: Metabolic / Endocrine  (Type 2 diabetes, Obesity, Thyroid disorders, Hyperlipidemia...)
4: Inflammatory / Immune  (Rheumatoid arthritis, Lupus, Atopic dermatitis, Crohn's disease...)
5: Respiratory  (Asthma, COPD, Pulmonary fibrosis, Bronchitis...)
6: Oncology / Cancer  (Lung cancer, Breast cancer, Colorectal cancer, Leukemia...)
7: Renal / Urological  (Chronic kidney disease, Overactive bladder, Glomerulonephritis...)
8: Gastrointestinal  (IBS, Gastroparesis, Ulcerative colitis, GERD, Hepatitis...)
9: Pain / Musculoskeletal  (Chronic pain, Osteoarthritis, Fibromyalgia, Neuropathic pain...)
10: Reproductive / Hormonal  (Endometriosis, Infertility, Preterm labor, Menopause...)
11: Infectious Disease  (HIV/AIDS, COVID-19, Tuberculosis, Viral/bacterial infections...)
12: Other / Unclassified  (Rare diseases, multi-system disorders, unclear conditions...)

Rules:
- Choose the PRIMARY therapeutic context most relevant to GPCR drug development.
- Use 12 for rare/orphan diseases or genuinely ambiguous multi-system disorders.
- Respond ONLY with JSON: {"disease_name": category_number}. No markdown, no text."""


def classify_batch_gemini(model, name_to_cui: dict) -> dict:
    """
    Send batch of {disease_name: cui} to Gemini.
    Returns {cui: category_id}.
    """
    disease_names = list(name_to_cui.keys())
    payload = json.dumps({"diseases": disease_names})
    prompt = f"{GEMINI_SYSTEM}\n\nInput: {payload}"
    try:
        resp = model.generate_content(prompt)
        raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", resp.text.strip())
        result = json.loads(raw)
        cui_to_cat = {}
        for name, cat in result.items():
            cui = name_to_cui.get(name)
            if cui:
                try:
                    cat_int = int(cat)
                    cui_to_cat[cui] = cat_int if 1 <= cat_int <= 12 else 12
                except (ValueError, TypeError):
                    cui_to_cat[cui] = 12
        return cui_to_cat
    except Exception as e:
        print(f"    [WARN] Gemini batch failed: {e}")
        return {cui: 12 for cui in name_to_cui.values()}


In [ ]:
# ── Main ─────────────────────────────────────────────────────────────────────
def main():
    raw_df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(raw_df):,} CUIs from Raw_Disease_List.csv")

    # Rows that can be classified (exclude INVALID/NOT_FOUND/Unknown)
    bad = {"NOT_FOUND", "INVALID_FORMAT", "Unknown", "Unknown disease"}
    classifiable = raw_df[~raw_df["disease_name"].isin(bad)].copy()
    unclassifiable = raw_df[raw_df["disease_name"].isin(bad)].copy()
    print(f"  Classifiable: {len(classifiable):,} | Unclassifiable: {len(unclassifiable):,}")

    results = {}   # {umls_cui: category_id}

    # ── Step 1: Rule-based keyword matching (instant) ─────────────────────────
    rule_hits = 0
    for _, row in classifiable.iterrows():
        cat = rule_based_category(str(row["disease_name"]))
        if cat is not None:
            results[row["umls_cui"]] = cat
            rule_hits += 1
    print(f"\nStep 1 (rule-based): {rule_hits:,} classified ({rule_hits/len(classifiable)*100:.1f}%)")

    # ── Step 2: Gemini for remaining ──────────────────────────────────────────
    remaining = classifiable[~classifiable["umls_cui"].isin(results)]
    print(f"Step 2 (Gemini):     {len(remaining):,} remaining")

    # Load checkpoint
    if Path(CHECKPOINT_PATH).exists():
        ckpt = pd.read_csv(CHECKPOINT_PATH)
        for _, row in ckpt.iterrows():
            if row["umls_cui"] not in results:
                results[row["umls_cui"]] = int(row["category_id"])
        remaining = remaining[~remaining["umls_cui"].isin(results)]
        print(f"  Checkpoint: {len(ckpt):,} loaded | still remaining: {len(remaining):,}")

    if len(remaining) > 0 and GOOGLE_API_KEY != "YOUR_GOOGLE_API_KEY_HERE":
        genai.configure(api_key=GOOGLE_API_KEY)
        model = genai.GenerativeModel(
            model_name="gemini-2.5-flash-lite",
            generation_config=genai.GenerationConfig(temperature=0.0),
        )
        batches = chunk_list(remaining.to_dict("records"), GEMINI_BATCH)
        ckpt_buffer = []

        for i, batch in enumerate(batches):
            name_to_cui = {row["disease_name"]: row["umls_cui"] for row in batch}
            cat_map = classify_batch_gemini(model, name_to_cui)
            results.update(cat_map)
            ckpt_buffer.extend([{"umls_cui": cui, "category_id": cat}
                                 for cui, cat in cat_map.items()])
            time.sleep(0.1)

            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(ckpt_buffer).to_csv(CHECKPOINT_PATH, index=False)
                print(f"  Batch {i+1}/{len(batches)} | checkpoint saved")

    elif len(remaining) > 0:
        print("  [SKIP] No API key — remaining will be category 12")
        for _, row in remaining.iterrows():
            results[row["umls_cui"]] = 12

    # ── Step 3: Assign fallback to unclassifiable ─────────────────────────────
    for _, row in unclassifiable.iterrows():
        results[row["umls_cui"]] = 12

    # ── Step 4: Build final DataFrame ─────────────────────────────────────────
    name_map = dict(zip(raw_df["umls_cui"], raw_df["disease_name"]))
    records = []
    for cui, cat_id in results.items():
        records.append({
            "umls_cui":      cui,
            "disease_name":  name_map.get(cui, ""),
            "category_id":   cat_id,
            "category_name": CATEGORIES.get(cat_id, "Other / Unclassified"),
        })

    final_df = pd.DataFrame(records).sort_values("category_id").reset_index(drop=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\n=== Category Distribution ===")
    dist = final_df.groupby(["category_id","category_name"]).size().reset_index(name="n")
    dist["pct"] = (dist["n"] / len(final_df) * 100).round(1)
    print(dist.to_string(index=False))
    print(f"\nTotal: {len(final_df):,} | Saved: {OUTPUT_PATH}")


In [ ]:
if __name__ == "__main__":
    main()


In [ ]:
"""
Post-processing: Fix misclassifications in UMLS_CUI_to_Category.csv
======================================================================
Known error patterns from QC:
  1. Pulmonary hypertension → was Cardiovascular, correct: Respiratory (5)
  2. Ocular hypertension / Glaucoma → was Cardiovascular, correct: Other (12)
  3. Nocardiosis → was Cardiovascular, correct: Infectious Disease (11)
  4. Retinal occlusion → was Cardiovascular, correct: Other (12)
  5. Renal vein thrombosis → was Cardiovascular, correct: Renal/Urological (7)
  
Strategy: apply disease-name keyword corrections AFTER Gemini classification.
This is a targeted override, not a full reclassification.
"""

import pandas as pd
import re

INPUT_PATH  = "./Output/DB/GPCRactDB/UMLS_CUI_to_Category.csv"
OUTPUT_PATH = "./Output/DB/GPCRactDB/UMLS_CUI_to_Category_v2.csv"

CATEGORIES = {
    1:  "CNS / Psychiatric",
    2:  "Cardiovascular",
    3:  "Metabolic / Endocrine",
    4:  "Inflammatory / Immune",
    5:  "Respiratory",
    6:  "Oncology / Cancer",
    7:  "Renal / Urological",
    8:  "Gastrointestinal",
    9:  "Pain / Musculoskeletal",
    10: "Reproductive / Hormonal",
    11: "Infectious Disease",
    12: "Other / Unclassified",
}

# Override rules: (keyword_pattern, correct_category_id)
# Applied only within specific source categories to minimize false positives.
OVERRIDE_RULES = [
    # Pulmonary hypertension is Respiratory, not Cardiovascular
    (r"pulmonary hypertension|pulmonary arterial|pulmonar.*hyperten", 5),
    # Ocular / ophthalmic → Other
    (r"ocular|intraocular|glaucoma|ophthalm|retinal|macular|vitreous", 12),
    # Nocardiosis, other bacterial infections → Infectious Disease
    (r"nocardiosis|nocardia", 11),
    # Retinal vascular → Other
    (r"retinal.*occlu|central retinal", 12),
    # Renal vein thrombosis → Renal
    (r"renal vein thrombosis|renal.*thrombo", 7),
]

def apply_overrides(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    corrections = 0

    for pattern, correct_id in OVERRIDE_RULES:
        mask = df["disease_name"].str.lower().str.contains(pattern, na=False, regex=True)
        wrong = mask & (df["category_id"] != correct_id)
        n_fixed = wrong.sum()
        if n_fixed > 0:
            print(f"  [{pattern[:40]:40s}] → cat {correct_id}: fixed {n_fixed} rows")
            df.loc[wrong, "category_id"] = correct_id
            df.loc[wrong, "category_name"] = CATEGORIES[correct_id]
            corrections += n_fixed

    print(f"\nTotal corrections: {corrections}")
    return df

def main():
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df):,} rows")
    print(f"\nBefore correction:")
    print(df["category_name"].value_counts().to_string())

    df_fixed = apply_overrides(df)

    print(f"\nAfter correction:")
    print(df_fixed["category_name"].value_counts().to_string())

    # Spot-check: verify nocardiosis is now Infectious Disease
    check = df_fixed[df_fixed["disease_name"].str.lower().str.contains("nocardiosis", na=False)]
    print(f"\nVerification - nocardiosis category: {check['category_name'].tolist()}")

    check2 = df_fixed[df_fixed["disease_name"].str.lower().str.contains("pulmonary hypertension", na=False)]
    print(f"Verification - pulmonary hypertension: {check2['category_name'].unique().tolist()}")

    df_fixed.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()
